<a href="https://colab.research.google.com/github/csu-techhub/quantum-optimization-simulation/blob/main/module34a/Lab02.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Lab 2 — Pauli Operators, Tensor Products, and the R$_{ZZ}$ Gate

**Maps to:** Module 3, Lesson 3 (Two-Qubit Rotation Gates)

**Time:** ~60 minutes (instructor walkthrough ~12 min)

---

### The question this lab answers

Module 3 asks: *is there a circuit that swaps $|10\rangle \leftrightarrow |01\rangle$
while leaving $|00\rangle$ and $|11\rangle$ alone?* The lecture tries $X\otimes X$, then
$Y\otimes Y$, then $X\otimes X + Y\otimes Y$ — and finds that the sum **is not a legal
gate**. The fix is to put it in an exponent.

Here you will verify every one of those matrices yourself, then build the first
exponential gate, R$_{ZZ}$, and confirm that the three-gate circuit
CNOT–R$_z$–CNOT really equals $e^{-i\frac{\theta}{2} Z\otimes Z}$.

### After this lab you can
1. Build two-qubit operators with `np.kron` and read the 4×4 matrices off the slides.
2. Explain *why* $XX+YY$ cannot be a gate, and check it numerically.
3. Turn any Pauli string $P$ (with $P^2=I$) into a gate via
   $e^{-i\frac{\theta}{2}P} = \cos\frac{\theta}{2}I - i\sin\frac{\theta}{2}P$.
4. Show that R$_{ZZ}(\theta)|ab\rangle = e^{-i\frac{\theta}{2}(-1)^{a\oplus b}}|ab\rangle$.
5. Compile R$_{ZZ}$ into CNOT–R$_z$–CNOT and verify it gate-for-gate.

In [ ]:
# %pip install -q qiskit qiskit-aer matplotlib
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector

np.set_printoptions(precision=3, suppress=True)

I2 = np.eye(2, dtype=complex)
X  = np.array([[0, 1], [1, 0]], dtype=complex)
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z  = np.array([[1, 0], [0, -1]], dtype=complex)

def is_unitary(U, tol=1e-10):
    U = np.asarray(U, dtype=complex)
    return np.allclose(U.conj().T @ U, np.eye(U.shape[0]), atol=tol)

def is_hermitian(M, tol=1e-10):
    M = np.asarray(M, dtype=complex)
    return np.allclose(M.conj().T, M, atol=tol)

print("X unitary?", is_unitary(X), " Y unitary?", is_unitary(Y), " Z unitary?", is_unitary(Z))
print("X^2 = I ?", np.allclose(X @ X, I2))
print("X @ Y  = i Z ?", np.allclose(X @ Y, 1j * Z))

## 1. Two qubits: the tensor (Kronecker) product

Two qubits live in a **4**-dimensional space with basis
$|00\rangle, |01\rangle, |10\rangle, |11\rangle$. An operator acting on both qubits is a
4×4 matrix built with $\otimes$ (`np.kron`):

$$ (A \otimes B)\,|q_0 q_1\rangle \;=\; A|q_0\rangle \otimes B|q_1\rangle .$$

`np.kron(A, B)` tiles a copy of `B` scaled by every entry of `A`. That is all it is.

> **Convention note.** In this lab we write basis states as $|q_0 q_1\rangle$
> (left = qubit 0), matching the lecture slides, and we build operators as
> `np.kron(op_on_q0, op_on_q1)`. Qiskit's own `SparsePauliOp("XZ")` uses the *opposite*
> reading (rightmost letter = qubit 0). For the symmetric operators in this lab
> ($XX$, $YY$, $ZZ$) the two conventions agree, so nothing here breaks — but flag it in
> your memory for Lab 4.

In [ ]:
XX = np.kron(X, X)
YY = np.kron(Y, Y)
ZZ = np.kron(Z, Z)

print("X (x) X =\n", XX.real)
print("\nY (x) Y =\n", YY.real, "  (all entries happen to be real)")
print("\nZ (x) Z =\n", ZZ.real)

Compare those two matrices with the slide. $X\otimes X$ has $1$'s on the anti-diagonal;
$Y\otimes Y$ has the same anti-diagonal but with **minus signs in the corners**.

### Exercise 1 — what does $XX$ actually *do* to each basis state?

Apply `XX` to each of the four basis vectors and identify the output state.

In [ ]:
basis = {"|00>": np.array([1,0,0,0], dtype=complex),
         "|01>": np.array([0,1,0,0], dtype=complex),
         "|10>": np.array([0,0,1,0], dtype=complex),
         "|11>": np.array([0,0,0,1], dtype=complex)}

def label_of(vec):
    '''Return the basis label of a vector that is +/-1 times a basis state.'''
    for name, b in basis.items():
        if np.allclose(np.abs(vec), np.abs(b)):
            sign = "-" if np.real(vec[np.argmax(np.abs(vec))]) < 0 else "+"
            return sign + name
    return "superposition"

for name, b in basis.items():
    # TODO: print the result of applying XX and of applying YY to this basis state
    ...

**Read the output carefully.** $XX$ maps $|10\rangle \to |01\rangle$ (good!) but it also
maps $|00\rangle \to |11\rangle$ (bad — we wanted those left alone). $YY$ does the same
swaps but attaches minus signs to the $|00\rangle\!\leftrightarrow\!|11\rangle$ pair.

That is precisely why the lecture adds them.

## 2. $XX + YY$: the right idea, an illegal gate

### Exercise 2 — reproduce the slide, then test whether it is a gate

In [ ]:
S = ...          # TODO: XX + YY
print("XX + YY =\n", S.real)

print("\nHermitian (a valid *observable*)?", is_hermitian(S))
print("Unitary   (a valid *gate*)?      ", is_unitary(S))
print("\nColumn norms:", np.round(np.linalg.norm(S, axis=0), 3))
assert is_hermitian(S) and not is_unitary(S)
print("\nPASS: XX+YY is a legitimate observable but NOT a legitimate gate.")

## 3. The rescue: put it in an exponent

For any Hermitian $P$ with $P^2 = I$ (every Pauli string qualifies),

$$ e^{-i\frac{\theta}{2}P} \;=\; \cos\!\Big(\frac{\theta}{2}\Big) I \;-\; i\,\sin\!\Big(\frac{\theta}{2}\Big) P ,$$

and this **is** unitary for every real $\theta$. It is the matrix version of Euler's
formula $e^{-i\phi} = \cos\phi - i\sin\phi$; the series works out because
$P^2 = I$ collapses all the even powers.

Two useful readings of the same object:
* $\theta$ is a **knob**. At $\theta=0$ you get the identity; turning it continuously
  deforms the state.
* Wherever $P$ has eigenvalue $+1$ you pick up phase $e^{-i\theta/2}$; where it has
  eigenvalue $-1$ you pick up $e^{+i\theta/2}$. A Pauli exponential is a
  **phase rotation conditioned on the eigenvalue of $P$**.

In [ ]:
import scipy.linalg as la

def pauli_exp(P, theta):
    '''exp(-i theta/2 P) computed from the closed form, for P Hermitian with P^2 = I.'''
    n = P.shape[0]
    return np.cos(theta/2) * np.eye(n) - 1j * np.sin(theta/2) * P

theta = 0.7
for name, P in [("ZZ", ZZ), ("XX", XX), ("YY", YY), ("XX+YY", XX+YY)]:
    closed = pauli_exp(P, theta)
    series = la.expm(-1j * theta/2 * P)          # brute-force matrix exponential
    ok_form = np.allclose(closed, series)
    print(f"{name:6s}  P^2 = I ? {np.allclose(P@P, np.eye(4))!s:5s} "
          f" closed form == expm ? {ok_form!s:5s}  unitary ? {is_unitary(series)}")

Note the last row: even though $XX+YY$ squares to $2(I - ZZ)$, **not** $I$ — so the
convenient $\cos/\sin$ formula fails — its exponential is still perfectly unitary.
Exponentiating *always* produces a legal gate from a Hermitian generator. We will use
this fact in Lab 3.

## 4. The R$_{ZZ}$ gate

$$\mathrm{R}_{ZZ}(\theta) \equiv e^{-i\frac{\theta}{2} Z\otimes Z}
= \operatorname{diag}\big(e^{-i\theta/2},\, e^{+i\theta/2},\, e^{+i\theta/2},\, e^{-i\theta/2}\big).$$

### Exercise 3 — confirm the "parity phase" rule

Show numerically that
$\mathrm{R}_{ZZ}(\theta)|ab\rangle = e^{-i\frac{\theta}{2}(-1)^{a \oplus b}}|ab\rangle$,
where $a\oplus b$ is XOR. In words: **states with even parity get one phase, odd parity
gets the other. Probabilities never change.**

In [ ]:
theta = 0.7
RZZ = pauli_exp(ZZ, theta)

print("RZZ(theta) =\n", np.round(RZZ, 3))
print()
for name, b in basis.items():
    a_bit, b_bit = int(name[1]), int(name[2])
    parity = ...            # TODO: XOR of the two bits  (Python XOR operator is ^)
    predicted = ...         # TODO: exp(-i theta/2 (-1)^parity)
    actual = (RZZ @ b)[np.argmax(np.abs(b))]
    print(f"{name}: parity={parity}  predicted phase={predicted:+.4f}  actual={actual:+.4f}"
          f"   match={np.isclose(predicted, actual)}")
    assert np.isclose(predicted, actual)
print("\nPASS")

## 5. From matrix to circuit: CNOT — R$_z(\theta)$ — CNOT

Hardware does not accept a 4×4 matrix. The standard compilation is

```
q0: ──■─────────────■──
      │             │
q1: ──⊕──[Rz(θ)]────⊕──
```

The first CNOT writes the **parity** $a\oplus b$ into qubit 1; `Rz` applies the
parity-dependent phase; the second CNOT undoes the bookkeeping. Three gates, one knob.

This matters practically: on IBM superconducting hardware `Rz` is a *virtual* gate — the
control electronics just shift the phase reference of later pulses — so it is essentially
free. The cost of this circuit is the two CNOTs.

### Exercise 4 — build it and prove it equals the matrix

In [ ]:
def rzz_circuit(theta):
    '''CNOT - Rz(theta) on q1 - CNOT.'''
    qc = QuantumCircuit(2, name="RZZ")
    # TODO: three lines -- qc.cx(control, target), qc.rz(angle, qubit), qc.cx(...)
    ...
    return qc

theta = 0.7
qc = rzz_circuit(theta)
print(qc.draw(output="text"))

U_circuit = Operator(qc).data
U_matrix  = pauli_exp(ZZ, theta)
print("\ncircuit unitary =\n", np.round(U_circuit, 3))
assert np.allclose(U_circuit, U_matrix), "circuit does not match the matrix!"
print("\nPASS: CNOT-Rz-CNOT == exp(-i theta/2 ZZ)")

qc_builtin = QuantumCircuit(2); qc_builtin.rzz(theta, 0, 1)
assert np.allclose(Operator(qc_builtin).data, U_matrix)
print("PASS: matches Qiskit's built-in qc.rzz as well")

### Exercise 5 — why R$_{ZZ}$ alone can never mix configurations

Prepare the superposition $\frac{1}{\sqrt2}(|01\rangle + |10\rangle)$, apply
R$_{ZZ}(\theta)$ for several $\theta$, and print the probabilities.

In [ ]:
psi0 = (basis["|01>"] + basis["|10>"]) / np.sqrt(2)

print(f"{'theta':>6} | {'P(01)':>7} {'P(10)':>7} | amplitudes")
for theta in [0.0, 0.5, 1.0, 2.0, np.pi]:
    out = pauli_exp(ZZ, theta) @ psi0
    p = np.abs(out)**2
    print(f"{theta:6.2f} | {p[1]:7.4f} {p[2]:7.4f} | {np.round(out,3)}")

print("\nThe probabilities NEVER move. RZZ is a pure phase rotation:")
print("it changes how amplitudes will later interfere, but on its own it")
print("cannot move an electron from one configuration to another.")

## 6. Checkpoint

1. `np.kron(X, I2)` — which qubit does it act on, and what does it do to $|01\rangle$?
2. Give one reason $XX + YY$ fails as a gate that a student could state in one sentence.
3. R$_{ZZ}(\theta)$ applied to $|01\rangle$ gives which phase, $e^{-i\theta/2}$ or
   $e^{+i\theta/2}$?
4. How many CNOTs does R$_{ZZ}$ cost? Why does that number, and not the R$_z$, dominate
   the error on real hardware?
5. R$_{ZZ}$ leaves all probabilities fixed. So what *has* to be different about
   R$_{XX}$ and R$_{YY}$ if they are going to mix $|10\rangle$ and $|01\rangle$?

### What is next
**Lab 3** builds R$_{XX}$ and R$_{YY}$ by sandwiching R$_{ZZ}$ between basis-change
gates, and shows that the product R$_{XX}(\theta)$R$_{YY}(\theta)$ does exactly the
mixing Module 3 asked for.